# Threat hunting playbook

Hypothesis-driven hunt on **IP reputation observations** (malicious and benign candidates) with a **separate IOC enrichment lookup**.

- **Main telemetry:** [NERD](https://nerd.liberouter.org/nerd/data/) `ip_rep.csv` via a **Generic HTTP API dataset provider** (`%%cribl_api`).
- **Enrichment:** [NERD `bad_ips.txt`](https://nerd.liberouter.org/nerd/data/bad_ips.txt) via Cribl Search **`externaldata`**, saved as a **lookup**, then **`join`** in KQL.

## Hunt charter

**Hypothesis:** IPs in NERD `ip_rep.csv` with high `reputation_score` overlap NERD’s high-confidence malicious list (`bad_ips.txt`); low-score rows and known resolver IPs act as benign candidates.

**Questions:**

1. Can we register `ip_rep.csv` as a Search **dataset** (Generic HTTP provider + REST)?
2. How many main-set IPs match the malicious enrichment lookup?
3. What does the score distribution look like for matched vs unmatched IPs?

**Demo IDs:** `notebook_nerd_http` (provider), `notebook_nerd_ips` (dataset), `notebook_nerd_malicious.csv` (lookup) — removed in **Cleanup**.

## Prerequisites

1. Hosted Cribl app with **Cribl Search** and **`%%cribl_api`** (`default_search`).
2. Search must reach `nerd.liberouter.org` (provider + `externaldata` fetch server-side — **no** `proxies.yml` changes in this app).
3. Lookups ≤ **10,000** rows; enrichment uses `| limit 10000` on `externaldata`.
4. Use **`timeout=`** and **`verbose=true`** on slow cells (HTTP dataset + `externaldata`).

**Related:** `Cribl_Search_Lookup_Magics.ipynb`, `Cribl_API_Examples.ipynb`, `Anomaly_Detection_PyOD.ipynb` (`externaldata` pattern).

### A) Generic HTTP dataset provider (NERD `ip_rep.csv`)

Creates `/m/default_search/search/dataset-providers` ([docs](https://docs.cribl.io/search/set-up-generic-http-api/)). Run **Cleanup** first if a prior run stopped early.

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_nerd_ips response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_nerd_http response=json

In [ ]:
%%cribl_api POST /m/default_search/search/dataset-providers var=nerd_provider response=json
json:
  type: api_http
  id: notebook_nerd_http
  description: Notebook threat hunt — NERD ip_rep.csv
  authenticationMethod: none
  availableEndpoints:
    - name: ip_reputation
      method: GET
      url: https://nerd.liberouter.org/nerd/data/ip_rep.csv
      headers: []
      dataField: ""

### B) Federated dataset

Registers **`notebook_nerd_ips`** with CSV breaker rules so `ip_address` and `reputation_score` parse as columns.

In [ ]:
%%cribl_api POST /m/default_search/search/datasets var=nerd_dataset response=json
json:
  type: api_http
  id: notebook_nerd_ips
  description: NERD IP reputation (notebook threat hunt)
  provider: notebook_nerd_http
  enabledEndpoints:
    - ip_reputation
  filter: "true"
  searchVersion: v1
  breakerRulesets:
    - CSV Datatypes
    - Cribl Search
  metadata:
    enableAcceleration: false

### C) Preview main dataset (mixed malicious / benign candidates)

Sample rows from the provider-backed dataset. Low `reputation_score` ≈ benign/low risk; high scores ≈ malicious activity in NERD.

In [ ]:
%%cribl_search var=main_preview lang=kql limit=2000 preview=true earliest=-50y latest=now timeout=300 verbose=true
dataset="notebook_nerd_ips"
| project ip_address, reputation_score
| sort by reputation_score desc
| limit 500

In [ ]:
import pandas as pd

def _pick_col(frame, *names):
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None

ip_col = _pick_col(main_preview, "ip_address", "ip")
score_col = _pick_col(main_preview, "reputation_score", "reputation", "score")

if ip_col is None:
    raise ValueError(f"Expected ip_address column; got {list(main_preview.columns)}")

main_ips_df = main_preview[[ip_col] + ([score_col] if score_col else [])].copy()
main_ips_df = main_ips_df.rename(columns={ip_col: "ip_address"})
if score_col:
    main_ips_df["reputation_score"] = pd.to_numeric(main_ips_df[score_col], errors="coerce")
    main_ips_df = main_ips_df.drop(columns=[score_col], errors="ignore")

BENIGN_RESOLVERS = ["1.1.1.1", "1.0.0.1", "8.8.8.8", "8.8.4.4"]
print("main preview rows:", len(main_ips_df))
if "reputation_score" in main_ips_df.columns:
    print("score min/max:", main_ips_df["reputation_score"].min(), main_ips_df["reputation_score"].max())
print("known benign resolvers (may not appear in NERD sample):", BENIGN_RESOLVERS)
main_ips_df.head(8)

### D) Enrichment lookup via `externaldata`

Fetch high-confidence malicious IPs ([`bad_ips.txt`](https://nerd.liberouter.org/nerd/data/bad_ips.txt)), normalize, and publish a Search lookup.

In [ ]:
%%cribl_search var=bad_ips_raw lang=kql limit=0 preview=false earliest=-50y latest=now timeout=300 verbose=true
externaldata
[
  "https://nerd.liberouter.org/nerd/data/bad_ips.txt"
]
with(
  datatype="CSV"
)
| limit 10000

In [ ]:
import re

IPV4_RE = re.compile(r"^\d{1,3}(?:\.\d{1,3}){3}$")

def _pick_col(frame, *names):
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None

ip_col = _pick_col(bad_ips_raw, "ip_address", "ip")
ips = []
if ip_col:
    ips = bad_ips_raw[ip_col].astype(str).str.strip().tolist()
else:
    raw_col = _pick_col(bad_ips_raw, "_raw", "event", "column1")
    if raw_col:
        for line in bad_ips_raw[raw_col].astype(str):
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if IPV4_RE.match(line):
                ips.append(line)
    else:
        for c in bad_ips_raw.columns:
            for v in bad_ips_raw[c].astype(str):
                v = v.strip()
                if IPV4_RE.match(v):
                    ips.append(v)

malicious_lookup_df = pd.DataFrame({
    "ip_address": sorted(set(ips)),
    "is_malicious": 1,
})
print("malicious lookup rows:", len(malicious_lookup_df))
malicious_lookup_df.head(10)

In [ ]:
%%cribl_save_search_lookup notebook_nerd_malicious.csv var=malicious_lookup_df replace=true

In [ ]:
%%cribl_search var=lookup_preview lang=kql limit=10 preview=true earliest=-50y latest=now
dataset="$vt_lookups" lookupFile="notebook_nerd_malicious"
| limit 10

### E) Threat hunt — `join` dataset with enrichment lookup

**Join** main `notebook_nerd_ips` to lookup `notebook_nerd_malicious` on `ip_address`. Use **`timeout=600`** for HTTP dataset + join; **`verbose=true`** shows progress in stdout.

In [ ]:
%%cribl_search var=hunt_hits lang=kql limit=500 preview=true earliest=-50y latest=now timeout=600 verbose=true
dataset="notebook_nerd_ips"
| project ip_address, reputation_score
| join kind=leftouter (
    dataset="$vt_lookups" lookupFile="notebook_nerd_malicious"
  ) on $left.ip_address == $right.ip_address
| where isnotnull(ip_address1)

### F) Interpret findings

In [ ]:
import matplotlib.pyplot as plt

n_hits = len(hunt_hits)
print("enriched malicious hits:", n_hits)

score_col = _pick_col(hunt_hits, "reputation_score", "reputation")
if score_col and n_hits:
    scores = pd.to_numeric(hunt_hits[score_col], errors="coerce").dropna()
    print("reputation_score among hits — min:", scores.min(), "max:", scores.max(), "median:", scores.median())
    scores.head(200).plot(kind="hist", bins=30, title="Reputation score (join hits)", color="crimson")
    plt.xlabel("reputation_score")
    plt.tight_layout()
    plt.show()
else:
    print("No score column or no hits — complete §A–D or use Fallback.")

if n_hits == 0:
    print("Verdict: INCONCLUSIVE — no join hits. Check provider, lookup, and column names.")
else:
    print(
        "Verdict: IPs in the main dataset match the NERD high-confidence malicious enrichment list. "
        "Review low-score rows in §C for benign candidates; NERD documents FP risk on blocklist use."
    )
hunt_hits.head(10)

In [ ]:
# ### Prompt:
# Summarize this threat-hunt for analysts.
#
# Join hits sample:
# {{ hunt_hits | describe }}
#
# Requirements:
# - explain main dataset vs enrichment lookup
# - note benign vs malicious interpretation using reputation_score
# - one recommended production hardening step

### Cleanup

In [ ]:
%%cribl_delete_search_lookup notebook_nerd_malicious.csv

In [ ]:
%%cribl_api DELETE /m/default_search/search/datasets/notebook_nerd_ips response=json

In [ ]:
%%cribl_api DELETE /m/default_search/search/dataset-providers/notebook_nerd_http response=json

### Fallback (no provider or NERD egress)

If provider POST fails, use `cribl_search_sample` and a tiny synthetic lookup — same join pattern:

```kusto
dataset=cribl_search_sample
| project ip_address = srcaddr
| join kind=leftouter (
    dataset="$vt_lookups" lookupFile="notebook_nerd_malicious"
  ) on $left.ip_address == $right.ip_address
```

### Next steps

- `Cribl_API_Examples.ipynb` — Search REST jobs
- `Cribl_Search_Lookup_Magics.ipynb` — lookup magics + `$vt_lookups`
- [NERD data](https://nerd.liberouter.org/nerd/data/) — refresh feeds hourly